In [1]:
import re
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

## Part I - Import and reorganization to homogene h5ad structure

In [2]:
lincs = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/LINCS/compound_perturbation_data.h5ad")

In [3]:
lincs.var["symbol"] = lincs.var_names.copy()
lincs.var_names = lincs.var["ensg_id"].astype(str).copy()
lincs.var = lincs.var.drop(columns=["ensg_id"])
lincs.var.index.name = "ensg_id" 

In [4]:
# remove nan genes
valid_gene_mask = ~pd.isna(lincs.var_names)
valid_gene_mask &= lincs.var_names.astype(str) != "nan"

lincs = lincs[:, valid_gene_mask].copy()

In [5]:
lincs.write_h5ad("/cluster/work/boeva/eheiss/datasets/LINCS/lincs.h5ad")

## Part II - Statistics

In [6]:
with open("/cluster/work/boeva/eheiss/scbFM/data/gene_list.txt") as f:
    gene_list = [line.strip() for line in f if line.strip()]

In [7]:
lincs = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/LINCS/lincs.h5ad")
print(lincs)

AnnData object with n_obs × n_vars = 121633 × 946
    obs: 'cell_id', 'det_plate', 'det_well', 'lincs_phase', 'pert_dose', 'pert_dose_unit', 'pert_id', 'pert_iname', 'pert_mfc_id', 'pert_time', 'pert_time_unit', 'pert_type', 'rna_plate', 'rna_well', 'condition', 'cell_type', 'dose', 'cov_drug_dose_name', 'cov_drug_name', 'control', 'canonical_smiles', 'SMILES', 'paired_control_index', 'cell_type_split_0', 'cell_type_split_1', 'cell_type_split_2', 'cell_type_split_3', 'cell_type_split_4', 'random_split_0', 'random_split_1', 'random_split_2', 'random_split_3', 'random_split_4', 'drug_split_0', 'drug_split_1', 'drug_split_2', 'drug_split_3', 'drug_split_4', 'cov_drug_dose_name_split_0', 'cov_drug_dose_name_split_1', 'cov_drug_dose_name_split_2', 'cov_drug_dose_name_split_3', 'cov_drug_dose_name_split_4'
    var: 'symbol'


In [9]:
gene_set = set(map(str, gene_list))
var_names = np.asarray(lincs.var_names.astype(str))

in_list_mask = np.array([g in gene_set for g in var_names], dtype=bool)
not_in_list_mask = ~in_list_mask
not_in_list_weights = not_in_list_mask.astype(np.float64)

chunk_size = 1000

sum_frac_nonzero = 0.0
sum_frac_reads = 0.0
n_obs_done = 0

for start in range(0, lincs.n_obs, chunk_size):
    end = min(start + chunk_size, lincs.n_obs)

    X_chunk = lincs.X[start:end]

    if sparse.issparse(X_chunk):
        X_chunk = X_chunk.tocsr()

        total_nonzero = np.asarray(X_chunk.getnnz(axis=1)).ravel()
        total_reads = np.asarray(X_chunk.sum(axis=1)).ravel()

        # Same as X_chunk[:, not_in_list_mask].sum(axis=1), but avoids sparse fancy indexing.
        not_in_list_reads = np.asarray(X_chunk @ not_in_list_weights).ravel()

        X_binary = X_chunk.copy()
        X_binary.data = np.ones_like(X_binary.data, dtype=np.float64)
        not_in_list_nonzero = np.asarray(X_binary @ not_in_list_weights).ravel()

        del X_binary

    else:
        X_chunk = np.asarray(X_chunk)

        total_nonzero = (X_chunk > 0).sum(axis=1)
        not_in_list_nonzero = (X_chunk[:, not_in_list_mask] > 0).sum(axis=1)

        total_reads = X_chunk.sum(axis=1)
        not_in_list_reads = X_chunk[:, not_in_list_mask].sum(axis=1)

    frac_nonzero_not_in_list = np.divide(
        not_in_list_nonzero,
        total_nonzero,
        out=np.zeros_like(total_nonzero, dtype=float),
        where=total_nonzero > 0,
    )

    frac_reads_not_in_list = np.divide(
        not_in_list_reads,
        total_reads,
        out=np.zeros_like(total_reads, dtype=float),
        where=total_reads > 0,
    )

    sum_frac_nonzero += frac_nonzero_not_in_list.sum()
    sum_frac_reads += frac_reads_not_in_list.sum()
    n_obs_done += end - start

    if start == 0 or n_obs_done % (10 * chunk_size) == 0 or end == lincs.n_obs:
        print(f"Processed {n_obs_done:,}/{lincs.n_obs:,} samples")

    del X_chunk

print("Average portion of non-zero genes NOT in gene_list:",
      sum_frac_nonzero / n_obs_done)

print("Average portion of total reads NOT in gene_list:",
      sum_frac_reads / n_obs_done)


Processed 1,000/121,633 samples
Processed 10,000/121,633 samples
Processed 20,000/121,633 samples
Processed 30,000/121,633 samples
Processed 40,000/121,633 samples
Processed 50,000/121,633 samples
Processed 60,000/121,633 samples
Processed 70,000/121,633 samples
Processed 80,000/121,633 samples
Processed 90,000/121,633 samples
Processed 100,000/121,633 samples
Processed 110,000/121,633 samples
Processed 120,000/121,633 samples
Processed 121,633/121,633 samples
Average portion of non-zero genes NOT in gene_list: 0.038065049442285614
Average portion of total reads NOT in gene_list: 0.0377991270783654


## Part III - filter to gene list

In [10]:
gene_list = [str(g) for g in gene_list]
gene_index = pd.Index(lincs.var_names.astype(str))

present_genes = [g for g in gene_list if g in gene_index]
missing = [g for g in gene_list if g not in gene_index]
present_src_idx = gene_index.get_indexer(present_genes)

print(f"LINCS genes present: {len(present_genes)} / {len(gene_list)}")
print(f"LINCS genes missing: {len(missing)}")

out_path = "/cluster/work/boeva/eheiss/datasets/LINCS/lincs.h5ad"
chunk_size = 1000
chunks = []

for start in range(0, lincs.n_obs, chunk_size):
    end = min(start + chunk_size, lincs.n_obs)
    X_chunk = lincs.X[start:end, :]

    if sparse.issparse(X_chunk):
        # CSC is safer for column selection on some SciPy builds.
        X_out = X_chunk.tocsc()[:, present_src_idx].tocsr()
    else:
        X_out = np.asarray(X_chunk)[:, present_src_idx]

    chunk = ad.AnnData(
        X=X_out,
        obs=lincs.obs.iloc[start:end].copy(),
        var=pd.DataFrame(
            index=pd.Index(present_genes, name=lincs.var_names.name),
        ),
    )
    chunks.append(chunk)

    print(f"Prepared {end:,}/{lincs.n_obs:,} samples")

lincs_filtered = ad.concat(chunks, axis=0, join="inner", merge="same")
lincs_filtered.var_names = pd.Index(present_genes, name=lincs.var_names.name)

lincs_filtered.write_h5ad(out_path)
print("Wrote:", out_path)
print("Shape:", lincs_filtered.shape)


LINCS genes present: 910 / 13004
LINCS genes missing: 12094
Prepared 1,000/121,633 samples
Prepared 2,000/121,633 samples
Prepared 3,000/121,633 samples
Prepared 4,000/121,633 samples
Prepared 5,000/121,633 samples
Prepared 6,000/121,633 samples
Prepared 7,000/121,633 samples
Prepared 8,000/121,633 samples
Prepared 9,000/121,633 samples
Prepared 10,000/121,633 samples
Prepared 11,000/121,633 samples
Prepared 12,000/121,633 samples
Prepared 13,000/121,633 samples
Prepared 14,000/121,633 samples
Prepared 15,000/121,633 samples
Prepared 16,000/121,633 samples
Prepared 17,000/121,633 samples
Prepared 18,000/121,633 samples
Prepared 19,000/121,633 samples
Prepared 20,000/121,633 samples
Prepared 21,000/121,633 samples
Prepared 22,000/121,633 samples
Prepared 23,000/121,633 samples
Prepared 24,000/121,633 samples
Prepared 25,000/121,633 samples
Prepared 26,000/121,633 samples
Prepared 27,000/121,633 samples
Prepared 28,000/121,633 samples
Prepared 29,000/121,633 samples
Prepared 30,000/121,6